1. Domain layer: الـ entities + الـ interfaces (ports)
2. Mock Decision Engine (stub)
3. SQLite Ticket Repository
4. Gmail Notification Service
5. Use Case اللي بيربط كلشي مع بعض (بما فيه الـ loop تبع ASK_FOR_MORE_INFO)

In [2]:
# ============================================================
# Cell 0 — تحميل كل الـ Secrets من أيقونة المفتاح 🔑
# ============================================================
from google.colab import userdata

api_key            = userdata.get('HF_API_KEY')
gmail_address      = userdata.get('GMAIL_ADDRESS')
gmail_app_password = userdata.get('GMAIL_APP_PASSWORD')

print("✅ كل الـ Secrets اتحملت بنجاح")

SecretNotFoundError: Secret GMAIL_APP_PASSWORD does not exist.

In [ ]:
# ============================================================
# Cell 1 — Domain Layer (entities + enums + interfaces/ports)
# ============================================================
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Optional


class ActionType(str, Enum):
    CREATE_TICKET = "CREATE_TICKET"
    ASK_FOR_MORE_INFO = "ASK_FOR_MORE_INFO"
    REJECT_REQUEST = "REJECT_REQUEST"
    FLAG_OUT_OF_SCOPE = "FLAG_OUT_OF_SCOPE"
    ESCALATE_TO_HUMAN = "ESCALATE_TO_HUMAN"


@dataclass(frozen=True)
class DecisionResult:
    """ناتج الـ decision engine لرسالة مستخدم واحدة."""
    action: ActionType
    confidence: float          # من 0.0 لـ 1.0
    reasoning: str


@dataclass(frozen=True)
class TicketRecord:
    """تذكرة (ticket) جاهزة للحفظ بالداتابيز — مع حالتها."""
    user_message: str
    action: str                # 🆕 الحالة اللي قررها المحرك (CREATE_TICKET, ASK_FOR_MORE_INFO, ...)
    confidence: float
    reasoning: str
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())
    id: Optional[int] = None   # بيتحدد بعد الحفظ بالـ DB


# ---- Ports (interfaces) — هاي اللي رح نحقنها (Dependency Injection) -------

class IDecisionEngine(ABC):
    @abstractmethod
    def decide(self, user_message: str) -> DecisionResult:
        ...


class ITicketRepository(ABC):
    @abstractmethod
    def save(self, ticket: TicketRecord) -> int:
        """بيحفظ التذكرة وبيرجع الـ id تبعها."""
        ...


class INotificationService(ABC):
    @abstractmethod
    def notify(self, ticket: TicketRecord) -> None:
        ...

print("✅ Cell 1 — Domain Layer جاهز")

In [ ]:
# ============================================================
# Cell 2 — LLM-based Decision Engine (الحقيقي، بدل الـ Mock)
# ============================================================
import json
import re
from openai import OpenAI


class LLMDecisionEngine(IDecisionEngine):
    """
    Decision engine حقيقي بيعتمد على LLM عبر Hugging Face router
    (OpenAI-compatible client). الـ client والـ model والـ system_prompt
    محقونين (injected) عشان نقدر نستبدلهم أو نعمل mock إلهم بسهولة بالاختبار.
    """

    def __init__(self, client: OpenAI, model: str, system_prompt: str):
        self._client = client
        self._model = model
        self._system_prompt = system_prompt

    def decide(self, user_message: str) -> DecisionResult:
        raw_output = self._call_llm(user_message)
        return self._parse_response(raw_output)

    def _call_llm(self, user_message: str) -> str:
        completion = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": self._system_prompt},
                {"role": "user", "content": json.dumps({"user_message": user_message}, ensure_ascii=False)},
            ],
        )
        return completion.choices[0].message.content

    def _parse_response(self, raw_output: str) -> DecisionResult:
        try:
            match = re.search(r"\{.*\}", raw_output, re.DOTALL)
            data = json.loads(match.group(0) if match else raw_output)

            return DecisionResult(
                action=ActionType(data["action"]),
                confidence=float(data.get("confidence", 0.5)),
                reasoning=str(data.get("reasoning", "")).strip(),
            )
        except (json.JSONDecodeError, KeyError, ValueError, AttributeError):
            # ما قدرنا نفهم رد الموديل بثقة -> أسلم قرار هو اعطاؤه لإنسان
            return DecisionResult(
                action=ActionType.ESCALATE_TO_HUMAN,
                confidence=0.0,
                reasoning=f"تعذّر تفسير رد النموذج بشكل موثوق: {raw_output[:200]}"
            )

print("✅ Cell 2 — Decision Engine جاهز")

In [ ]:
# ============================================================
# Cell 3 — System prompt + إنشاء المحرك + رسائل الاختبار
# ============================================================

SYSTEM_PROMPT = """
انت Decision Engine لنظام تذاكر يتعامل مع مشاكل عقود قانونية فقط.
مهمتك الوحيدة: تقرر الإجراء التالي المناسب لرسالة المستخدم، مش تقديم استشارة قانونية.

اختر إجراء واحد فقط من هاي القائمة الثابتة:
- CREATE_TICKET: المشكلة واضحة ومحددة ومتعلقة بعقد ويمكن فتح تذكرة لمتابعتها.
- ASK_FOR_MORE_INFO: الرسالة غامضة أو ناقصة تفاصيل أساسية لاتخاذ قرار.
- REJECT_REQUEST: الطلب غير مرتبط بمشكلة عقد يمكن التعامل معها.
- FLAG_OUT_OF_SCOPE: الطلب خارج نطاق مشاكل العقود تمامًا (طبي، جنائي، إلخ).
- ESCALATE_TO_HUMAN: فيه خطر قانوني حقيقي أو حالة حساسة/طارئة تحتاج تدخل إنسان.

رجّع الجواب حصرًا بصيغة JSON زي هيك، بدون أي نص إضافي قبله أو بعده:
{"action": "...", "confidence": 0.0-1.0, "reasoning": "سطر أو سطرين بس"}
"""

# Initialize the OpenAI client
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=api_key,
)

real_engine = LLMDecisionEngine(
    client=client,
    model="Qwen/Qwen3-Coder-Next:novita",
    system_prompt=SYSTEM_PROMPT,
)

# ---- رسائل الاختبار ----
test_messages = [
    "عندي عقد إيجار والمالك غيّر بند التأمين بدون إشعار، بدي اعرف حقوقي.",   # -> CREATE_TICKET
    "في مشكلة بعقد ما بعرف كيف اوصفها.",                                      # -> ASK_FOR_MORE_INFO
    "بدي وصفة كبسة.",                                                          # -> FLAG_OUT_OF_SCOPE
    "الشركة رح تقاضيي وبتحصلون خسارة كبيرة!",                                  # -> ESCALATE_TO_HUMAN
]

print("✅ Cell 3 — المحرك ورسائل الاختبار جاهزين")

In [ ]:
# ============================================================
# Cell 4 — SQLite Repository Implementation
# ============================================================
import sqlite3


class SQLiteTicketRepository(ITicketRepository):
    def __init__(self, db_path: str = "tickets.db"):
        self.db_path = db_path
        self._create_table()

    def _create_table(self):
        """إنشاء الجدول إذا لم يكن موجوداً — عمود action مخصص للحالات"""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS tickets (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    user_message TEXT,
                    action TEXT,
                    confidence REAL,
                    reasoning TEXT,
                    created_at TEXT
                )
            """)

    def save(self, ticket: TicketRecord) -> int:
        """حفظ التذكرة — الحالة بتاخد من جوا التذكرة نفسها (مش يدوي)"""
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                "INSERT INTO tickets (user_message, action, confidence, reasoning, created_at) VALUES (?, ?, ?, ?, ?)",
                (ticket.user_message, ticket.action, ticket.confidence, ticket.reasoning, ticket.created_at)
            )
            return cursor.lastrowid

    def get_all_tickets(self):
        """كل الصفوف — كل الحالات"""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            return [dict(row) for row in conn.execute("SELECT * FROM tickets").fetchall()]

    def get_by_action(self, action: ActionType):
        """🆕 فلترة حسب الحالة — مثال: جيب بس التذاكر المنشأة (CREATE_TICKET)"""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                "SELECT * FROM tickets WHERE action = ?", (action.value,)
            ).fetchall()
            return [dict(row) for row in rows]

print("✅ Cell 4 — SQLite Repository جاهز")

In [ ]:
# ============================================================
# Cell 5 — Email Notification (إشعار Gmail لما تتفتح تذكرة)
# ============================================================
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


class EmailNotificationService(INotificationService):
    """بيبعت إيميل على Gmail لما تذكرة جديدة تتسجل."""

    def __init__(self, sender_email: str, sender_password: str, recipient_email: str):
        self._sender = sender_email          # البريد اللي بيبعت
        self._password = sender_password     # الـ App Password (مش كلمة السر العادية!)
        self._recipient = recipient_email    # البريد اللي بيستقبل

    def notify(self, ticket: TicketRecord) -> None:
        msg = MIMEMultipart("alternative")
        msg["From"] = self._sender
        msg["To"] = self._recipient
        msg["Subject"] = f"🎫 تذكرة جديدة #{ticket.id} — {ticket.action}"

        html = f"""
        <html><body style="font-family:Arial; direction:rtl">
          <h2 style="color:#2c7be5">🎫 تم إنشاء تذكرة جديدة</h2>
          <table border="1" cellpadding="8" style="border-collapse:collapse">
            <tr><td><b>رقم التذكرة</b></td><td>#{ticket.id}</td></tr>
            <tr><td><b>الحالة</b></td><td>{ticket.action}</td></tr>
            <tr><td><b>الثقة</b></td><td>{ticket.confidence}</td></tr>
            <tr><td><b>السبب</b></td><td>{ticket.reasoning}</td></tr>
            <tr><td><b>رسالة العميل</b></td><td>{ticket.user_message}</td></tr>
            <tr><td><b>التاريخ</b></td><td>{ticket.created_at}</td></tr>
          </table>
        </body></html>
        """
        msg.attach(MIMEText(html, "html", "utf-8"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(self._sender, self._password)
            server.send_message(msg)

print("✅ Cell 5 — خدمة الإشعارات جاهزة")

In [ ]:
# ============================================================
# Cell 8 — التشغيل: حفظ بالداتابيز + إشعار إيميل
# ============================================================

repo = SQLiteTicketRepository()

notifier = EmailNotificationService(
    sender_email=gmail_address,
    sender_password=gmail_app_password,
    recipient_email=gmail_address,   # بيبعتلك لنفسك — ممكن تحط أي إيميل تاني
)

for msg in test_messages:
    result = real_engine.decide(msg)
    print(f"Processing: {msg}")

    # حفظ في الداتابيز (كل الحالات)
    ticket = TicketRecord(
        user_message=msg,
        action=result.action.value,
        confidence=result.confidence,
        reasoning=result.reasoning
    )
    ticket_id = repo.save(ticket)
    print(f"  [Saved] #{ticket_id} | action={ticket.action}")

    # 📧 إشعار بس لما تتفتح تذكرة فعلاً
    if result.action == ActionType.CREATE_TICKET:
        try:
            notifier.notify(ticket)
            print(f"  [Email] 📧 إشعار اتبعت على {gmail_address}")
        except Exception as e:
            print(f"  [Warning] الإيميل فشل: {e}")

    # (اختياري) لو عايز إشعار كمان للحالات الطارئة ضيف:
    # elif result.action == ActionType.ESCALATE_TO_HUMAN: ...
    print("-" * 60)

# ---- عرض النتائج ----
import pandas as pd
df_tickets = pd.DataFrame(repo.get_all_tickets())
display(df_tickets)